# Project: Model Selection and Cross-Validation Strategies

Compare different cross-validation strategies and model selection techniques for robust performance estimation.

In [ ]:
import sys, os, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_PATH = '/content/drive/MyDrive/data-science-mastery'
    if os.path.isdir(REPO_PATH):
        os.chdir(REPO_PATH)
        print(f'Working directory: {os.getcwd()}')
        if not os.path.isdir('Data') or len(os.listdir('Data')) < 5:
            subprocess.check_call([sys.executable, 'scripts/prepare_datasets.py'])
    else:
        REPO_PATH = '/content/data-science-mastery'
        if os.path.isdir(REPO_PATH):
            os.chdir(REPO_PATH)
        else:
            print('Repo not found. Run opencolab_setup.ipynb first.')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import (cross_val_score, KFold, StratifiedKFold,
                                       RepeatedKFold, LeaveOneOut, ShuffleSplit,
                                       learning_curve, validation_curve)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
print('Libraries loaded')

In [ ]:
X, y = make_classification(n_samples=1000, n_features=20, n_informative=10,
                            n_redundant=5, n_classes=2, random_state=42)
print(f'Dataset: {X.shape[0]} samples, {X.shape[1]} features')

## Compare CV Strategies

In [ ]:
cv_strategies = {
    'KFold (k=5)': KFold(5, shuffle=True, random_state=42),
    'Stratified KFold (k=5)': StratifiedKFold(5, shuffle=True, random_state=42),
    'Repeated KFold (5x3)': RepeatedKFold(n_splits=5, n_repeats=3, random_state=42),
    'ShuffleSplit (100/20%)': ShuffleSplit(n_splits=10, test_size=0.2, random_state=42),
}
model = RandomForestClassifier(n_estimators=100, random_state=42)
results = []
for name, cv in cv_strategies.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
    results.append({'CV Strategy': name, 'Mean': scores.mean(),
                    'Std': scores.std(), 'Min': scores.min(), 'Max': scores.max()})
print(pd.DataFrame(results).round(4).to_string(index=False))

## Validation Curves

In [ ]:
param_range = [1, 5, 10, 20, 50, 100, 200]
train_scores, test_scores = validation_curve(
    RandomForestClassifier(random_state=42), X, y,
    param_name='n_estimators', param_range=param_range,
    cv=5, scoring='accuracy', n_jobs=-1)
plt.figure(figsize=(10, 5))
plt.plot(param_range, train_scores.mean(axis=1), 'o-', label='Train', color='blue')
plt.plot(param_range, test_scores.mean(axis=1), 'o-', label='Validation', color='red')
plt.fill_between(param_range, train_scores.mean(axis=1)-train_scores.std(axis=1),
                 train_scores.mean(axis=1)+train_scores.std(axis=1), alpha=0.1, color='blue')
plt.fill_between(param_range, test_scores.mean(axis=1)-test_scores.std(axis=1),
                 test_scores.mean(axis=1)+test_scores.std(axis=1), alpha=0.1, color='red')
plt.xlabel('n_estimators')
plt.ylabel('Accuracy')
plt.title('Validation Curve: Random Forest')
plt.xscale('log')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

- Compared KFold, StratifiedKFold, RepeatedKFold
- Validation curves for hyperparameter selection
- Cross-validation strategies for robust estimation